# Volume e Princípio de Cavalieri

**Módulo 06 — Geometria Espacial** · Notebook 09 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0609a_volume_e_principio_de_cavalieri.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (prismas, paralelepípedos, altura e seções) e [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (área da projeção)
- Módulo 04 — Geometria Plana: [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (áreas de polígonos)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- explicar o que é o volume de um sólido e converter unidades de volume e de capacidade;
- calcular o volume do paralelepípedo retângulo e do cubo, e comparar volumes de paralelepípedos;
- enunciar e aplicar o Princípio de Cavalieri;
- calcular o volume de qualquer prisma e resolver problemas práticos com volumes.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math

## Qual pilha tem mais papel?

Pegue um baralho e faça duas pilhas iguais, com o mesmo número de cartas. A primeira fica bem certinha, reta. A segunda você empurra para o lado, deixando-a inclinada, ou gira um pouco cada carta, deixando a pilha torcida, como uma escada em caracol.

A segunda pilha parece diferente, maior, mais "espalhada". Mas ela tem mais papel do que a primeira?

Claro que não: são as **mesmas cartas**! Cada carta tem a mesma área e a mesma espessura, então as duas pilhas ocupam o **mesmo volume**.

Essa observação simples tem nome: **Princípio de Cavalieri**. E ela vai permitir calcular volumes de sólidos muito mais complicados do que pilhas de cartas.

🕹️ Monte as duas pilhas no widget abaixo.

In [ ]:
def montar_pilhas(numero_de_cartas: int, inclinacao: float, torcao: float) -> None:
    """Desenha uma pilha reta e uma pilha inclinada e torcida, com as mesmas cartas."""
    # Cada carta é um quadrado de lado 2 (área 4); as duas pilhas têm altura 4.
    carta = u3d.poligono_regular(4, lado=2, giro=45)
    # A segunda pilha fica 5 unidades ao lado da primeira, só para desenhar ao lado.
    carta_deslocada = u3d.poligono_regular(4, lado=2, centro=(5, 0, 0), giro=45)

    pilha_1 = u3d.fatias_de_prisma(carta, (0, 0, 4), numero_de_cartas)
    pilha_2 = u3d.fatias_de_prisma(
        carta_deslocada, (inclinacao, 0, 4), numero_de_cartas, torcao=torcao
    )

    espessura = 4 / numero_de_cartas
    area_da_carta = 2 * 2
    volume_pilha_1 = numero_de_cartas * area_da_carta * espessura
    volume_pilha_2 = numero_de_cartas * area_da_carta * espessura

    fig = u3d.criar_figura(titulo="Duas pilhas com as mesmas cartas")
    fig = u3d.desenhar_pilha(fig, pilha_1, espessura)
    fig = u3d.desenhar_pilha(fig, pilha_2, espessura)
    fig = u3d.adicionar_rotulo(fig, (0, 0, 4.7), "pilha reta")
    fig = u3d.adicionar_rotulo(fig, (5 + inclinacao, 0, 4.7), "pilha torta")
    u3d.mostrar(fig)

    print(f"Espessura de cada carta: 4 / {numero_de_cartas} = {espessura:.3f}")
    print(
        f"Pilha reta:  {numero_de_cartas} cartas × área {area_da_carta} × "
        f"espessura {espessura:.3f} = {volume_pilha_1:.2f}"
    )
    print(
        f"Pilha torta: {numero_de_cartas} cartas × área {area_da_carta} × "
        f"espessura {espessura:.3f} = {volume_pilha_2:.2f}"
    )
    print("Mesmas cartas, mesmo volume, não importa a forma da pilha!")


interact(
    montar_pilhas,
    numero_de_cartas=widgets.IntSlider(
        min=5, max=60, step=5, value=20, description="cartas",
        continuous_update=False,
    ),
    inclinacao=widgets.FloatSlider(
        min=0, max=3, step=0.5, value=1.5, description="inclinação",
        continuous_update=False,
    ),
    torcao=widgets.FloatSlider(
        min=0, max=180, step=15, value=90, description="torção (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=20, continuous_update=False, description='cartas', max=60, min=5, step=5…

> 💡 **Dica.** Prédios torcidos, como o **Turning Torso**, em Malmö (Suécia), cujos andares vão girando até completar 90° do térreo ao topo, funcionam como a sua segunda pilha: se cada andar tiver a mesma área do andar de baixo, o volume do prédio é o mesmo de um prédio reto com os mesmos andares.

Antes de chegar ao Princípio de Cavalieri, precisamos responder a uma pergunta mais básica: **o que é volume** e como medi-lo?

## 1. O que é volume

Medir um comprimento é comparar com uma régua; medir uma área é contar quantos quadradinhos cabem numa figura. Medir um **volume** é contar quantos **cubinhos** cabem num sólido.

> 📌 **Definição.** O **volume** de um sólido é um número positivo que indica quantas vezes uma **unidade de volume** (um cubo de aresta 1) "cabe" no sólido. Ele obedece a três regras:
> 1. sólidos congruentes têm volumes iguais;
> 2. se um sólido é dividido em partes (sem sobreposição), o volume total é a soma dos volumes das partes;
> 3. o cubo de aresta 1 tem volume 1.

### Unidades de volume e de capacidade

| Unidade | Cubo de aresta | Equivalência |
|---|---|---|
| $1 \text{ m}^3$ | 1 m | $1000 \text{ dm}^3$ = **1000 litros** |
| $1 \text{ dm}^3$ | 10 cm | $1000 \text{ cm}^3$ = **1 litro** |
| $1 \text{ cm}^3$ | 1 cm | $1000 \text{ mm}^3$ = **1 mililitro** |

> ⚠️ **Erro comum.** Pensar que $1 \text{ m}^3 = 10 \text{ dm}^3$ ou $100 \text{ dm}^3$. Um metro tem 10 decímetros, mas um **cubo** de 1 m de aresta tem $10 \times 10 \times 10 = 1000$ cubinhos de 1 dm. Em volume, cada passo da tabela multiplica por **1000**.

🕹️ Monte uma caixa com cubinhos e conte quantos cabem. Use o controle "camadas" para montar a caixa aos poucos.

In [3]:
def montar_caixa(comprimento: int, largura: int, altura: int, camadas: int) -> None:
    """Monta uma caixa com cubinhos de aresta 1, camada por camada, e conta os cubinhos."""
    # Cada cubinho é guardado pelo seu canto de baixo. Os três laços percorrem a
    # caixa como quem empilha: fileira por fileira, camada por camada.
    origens = []
    for z in range(min(camadas, altura)):
        for y in range(largura):
            for x in range(comprimento):
                origens.append((x, y, z))

    caixa = u3d.paralelepipedo_retangulo(comprimento, largura, altura)
    fig = u3d.criar_figura(titulo="Contando cubinhos")
    fig = u3d.desenhar_poliedro(fig, *caixa, opacidade=0.05)
    fig = u3d.desenhar_blocos(fig, origens)
    u3d.mostrar(fig)

    print(f"Cubinhos em cada camada: {comprimento} × {largura} = {comprimento * largura}")
    print(f"Camadas montadas: {min(camadas, altura)}")
    print(f"Total de cubinhos: {len(origens)}")
    if camadas >= altura:
        print(
            f"Volume = {comprimento} × {largura} × {altura} = "
            f"{comprimento * largura * altura} unidades de volume"
        )


def contador(descricao: str, minimo: int, maximo: int, valor: int) -> widgets.IntSlider:
    """Cria um slider inteiro, de 1 em 1."""
    return widgets.IntSlider(
        min=minimo, max=maximo, step=1, value=valor, description=descricao,
        continuous_update=False,
    )


interact(
    montar_caixa,
    comprimento=contador("comprimento", 1, 6, 4),
    largura=contador("largura", 1, 5, 3),
    altura=contador("altura", 1, 5, 2),
    camadas=contador("camadas", 0, 5, 5),
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='comprimento', max=6, min=1), In…

**Pare e pense**

1. Quantos litros cabem numa caixa d'água de 1 m³?
2. Uma lata de refrigerante tem 350 mL. Quantos cm³ é isso?

<details>
<summary>Ver respostas</summary>

1. 1000 litros.
2. 350 cm³ (1 mL = 1 cm³).
</details>

## 2. Volume do paralelepípedo retângulo e do cubo

### Razão entre paralelepípedos retângulos

**Teorema**
> Se dois paralelepípedos retângulos têm duas dimensões iguais, a razão entre os seus volumes é igual à razão entre as terceiras dimensões.

Em outras palavras: mantendo comprimento e largura, se a altura dobra, o volume dobra; se a altura triplica, o volume triplica. Quando as alturas são números inteiros (ou frações), isso se vê contando camadas de cubinhos, como no widget anterior. Para alturas irracionais, como $\sqrt{2}$, a demonstração usa aproximações cada vez melhores por frações.

Aplicando o teorema três vezes, uma dimensão de cada vez, chegamos a uma consequência importante:
> A razão entre os volumes de dois paralelepípedos retângulos é
> $$\frac{V_1}{V_2} = \frac{a_1 \cdot b_1 \cdot c_1}{a_2 \cdot b_2 \cdot c_2}$$

### A fórmula do volume

Comparando um paralelepípedo retângulo de dimensões $a$, $b$, $c$ com o cubo de aresta 1 (volume 1):
$$\frac{V}{1} = \frac{a \cdot b \cdot c}{1 \cdot 1 \cdot 1} \quad\Longrightarrow\quad \boxed{V = a \cdot b \cdot c}$$

Como $a \cdot b$ é a área da base, também podemos escrever:
$$V = (\text{área da base}) \times (\text{altura})$$

**Cubo** de aresta $a$: $V = a^3$. (É por isso que se diz "$a$ **ao cubo**"!)

> ⚠️ **Erro comum.** Achar que dobrar a aresta de um cubo dobra o volume. Dobrando a aresta, **cada uma** das três dimensões dobra, e o volume fica multiplicado por $2 \times 2 \times 2 = 8$.

### E se as medidas não forem inteiras?

🕹️ O widget abaixo monta uma caixa com cubinhos menores, de aresta $\frac{1}{2}$ ou $\frac{1}{4}$. Cada cubinho tem volume $\left(\frac{1}{2}\right)^3 = \frac{1}{8}$ ou $\left(\frac{1}{4}\right)^3 = \frac{1}{64}$, e a conta confirma a fórmula $V = abc$.

In [ ]:
NOMES_DOS_CUBINHOS = {1: "1", 0.5: "1/2", 0.25: "1/4"}


def montar_caixa_fracionaria(
    tamanho_do_cubinho: float, comprimento: float, largura: float, altura: float
) -> None:
    """Preenche a caixa com cubinhos do tamanho escolhido e compara com V = abc."""
    # Uma medida é múltipla do tamanho do cubinho quando a divisão dá um número
    # inteiro (com uma pequena folga para os arredondamentos do computador).
    for medida in (comprimento, largura, altura):
        divisao = medida / tamanho_do_cubinho
        if abs(divisao - round(divisao)) > 1e-9:
            print("Escolha medidas que sejam múltiplas do tamanho do cubinho.")
            return

    n_x = round(comprimento / tamanho_do_cubinho)
    n_y = round(largura / tamanho_do_cubinho)
    n_z = round(altura / tamanho_do_cubinho)
    quantidade = n_x * n_y * n_z
    volume_do_cubinho = tamanho_do_cubinho ** 3
    volume_por_contagem = quantidade * volume_do_cubinho
    volume_pela_formula = comprimento * largura * altura

    # Os cantos dos cubinhos andam de tamanho_do_cubinho em tamanho_do_cubinho.
    origens = []
    for k in range(n_z):
        for j in range(n_y):
            for i in range(n_x):
                origens.append(
                    (i * tamanho_do_cubinho, j * tamanho_do_cubinho, k * tamanho_do_cubinho)
                )

    fig = u3d.criar_figura(titulo=f"Cubinhos de aresta {NOMES_DOS_CUBINHOS[tamanho_do_cubinho]}")
    fig = u3d.desenhar_blocos(fig, origens, aresta=tamanho_do_cubinho)
    u3d.mostrar(fig)

    nome = NOMES_DOS_CUBINHOS[tamanho_do_cubinho]
    print(f"Quantidade de cubinhos: {n_x} × {n_y} × {n_z} = {quantidade}")
    print(f"Volume de cada cubinho: ({nome})³ = {volume_do_cubinho:g}")
    print(f"Volume por contagem: {quantidade} × {volume_do_cubinho:g} = {volume_por_contagem:g}")
    print(
        f"Volume pela fórmula: {comprimento:g} × {largura:g} × {altura:g} = "
        f"{volume_pela_formula:g}"
    )


def medida(descricao: str, valor: float) -> widgets.FloatSlider:
    """Cria um slider de 0,5 a 3, de 0,25 em 0,25."""
    return widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=valor, description=descricao,
        continuous_update=False,
    )


interact(
    montar_caixa_fracionaria,
    tamanho_do_cubinho=widgets.Dropdown(
        options=[("1", 1), ("1/2", 0.5), ("1/4", 0.25)], value=0.25,
        description="cubinho",
    ),
    comprimento=medida("comprimento", 1.5),
    largura=medida("largura", 1),
    altura=medida("altura", 0.75),
);

interactive(children=(Dropdown(description='cubinho', index=2, options=(('1', 1), ('1/2', 0.5), ('1/4', 0.25))…

In [5]:
# Partimos de um paralelepípedo 2 × 3 × 4 e dobramos as dimensões, uma de cada vez.
casos = [
    ("original", 2, 3, 4),
    ("dobrando a altura", 2, 3, 8),
    ("dobrando altura e largura", 2, 6, 8),
    ("dobrando as três dimensões", 4, 6, 8),
]
volume_original = 2 * 3 * 4

print(f"{'caso':<28}{'a × b × c':>11}{'volume':>8}{'volume / volume original':>27}")
for nome, a, b, c in casos:
    volume = a * b * c
    dimensoes = f"{a} × {b} × {c}"
    print(f"{nome:<28}{dimensoes:>11}{volume:>8}{volume / volume_original:>27g}")

print()
print(
    "Dobrar as três dimensões multiplica o volume por 8. "
    "Você vai rever isso no notebook 14, sobre sólidos semelhantes."
)

caso                          a × b × c  volume   volume / volume original
original                      2 × 3 × 4      24                          1
dobrando a altura             2 × 3 × 8      48                          2
dobrando altura e largura     2 × 6 × 8      96                          4
dobrando as três dimensões    4 × 6 × 8     192                          8

Dobrar as três dimensões multiplica o volume por 8. Você vai rever isso no notebook 14, sobre sólidos semelhantes.


**Pare e pense**

1. Qual é o volume de um cubo de aresta 3 cm?
2. Um paralelepípedo retângulo tem volume 60 cm³. Qual será o volume se o comprimento triplicar e as outras medidas continuarem iguais?

<details>
<summary>Ver respostas</summary>

1. $3^3 = 27$ cm³.
2. 180 cm³ (pelo teorema da razão: triplicando uma dimensão, o volume triplica).
</details>

## 3. O Princípio de Cavalieri

Volte às pilhas de cartas do gancho. O que garante que elas têm o mesmo volume? Em **qualquer altura**, a carta de uma pilha e a carta da outra têm a **mesma área**.

**Princípio de Cavalieri**
> Considere dois sólidos e um plano $\alpha$. Se **todo** plano paralelo a $\alpha$ que corta um dos sólidos também corta o outro, e as duas seções obtidas têm **áreas iguais**, então os dois sólidos têm **volumes iguais**.

Pense nos sólidos como pilhas de fatias finíssimas. Se as fatias correspondentes têm sempre a mesma área (e a mesma espessura), as duas pilhas têm o mesmo volume, mesmo que as fatias tenham formas diferentes ou estejam em posições diferentes.

> ⚠️ **Erro comum.**
> - Achar que basta as **bases** terem a mesma área. O princípio exige áreas iguais em **todas** as alturas. Uma pirâmide e um prisma com a mesma base e a mesma altura **não** têm o mesmo volume: subindo, as seções da pirâmide vão encolhendo (você verá isso no notebook 10).
> - Achar que vale a volta. Dois sólidos podem ter o mesmo volume sem ter seções de mesma área em todas as alturas.

> 🕵️ **Curiosidade histórica.** O italiano **Bonaventura Cavalieri**, aluno de Galileu, publicou esse princípio em 1635, imaginando os sólidos como formados por infinitas fatias "indivisíveis". Mais de mil anos antes, na China, o matemático **Zu Gengzhi** (séculos V e VI) usou a mesma ideia para calcular o volume da esfera. Por isso, na China, ele é chamado de "princípio de Zu Geng".

🕹️ **Veja o princípio funcionar com fatias.** O widget abaixo compara um prisma reto com um prisma oblíquo e torcido, de mesma base e mesma altura. Escolha a altura de corte e compare as duas seções. Depois, aumente o número de fatias e veja a soma das fatias.

In [6]:
BASE_HEXAGONAL = u3d.poligono_regular(6, lado=1)
ALTURA_DOS_SOLIDOS = 4
AFASTAMENTO = 4  # o sólido 2 fica 4 unidades ao lado do sólido 1, só para desenhar


def comparar_solidos(
    altura_do_corte: float, inclinacao: float, torcao: float, numero_de_fatias: int,
    mostrar_fatias: bool,
) -> None:
    """Compara as seções e as somas de fatias de um prisma reto e de um torcido."""
    base_1 = BASE_HEXAGONAL
    base_2 = u3d.poligono_regular(6, lado=1, centro=(AFASTAMENTO, 0, 0))
    topo_1 = (0, 0, ALTURA_DOS_SOLIDOS)  # sólido 1: prisma reto
    topo_2 = (inclinacao, 0, ALTURA_DOS_SOLIDOS)  # sólido 2: inclinado e torcido

    # As seções na altura escolhida. Uma pilha de 16 fatias de espessura 0,25
    # (16 × 0,25 = 4) tem a parte de baixo da fatia k exatamente na altura k × 0,25:
    # esse polígono é a seção do sólido nessa altura. Por isso o controle da altura
    # anda de 0,25 em 0,25.
    k = round(altura_do_corte / 0.25)
    secao_1 = u3d.fatias_de_prisma(base_1, topo_1, 16)[k]
    secao_2 = u3d.fatias_de_prisma(base_2, topo_2, 16, torcao=torcao)[k]

    fatias_1 = u3d.fatias_de_prisma(base_1, topo_1, numero_de_fatias)
    fatias_2 = u3d.fatias_de_prisma(base_2, topo_2, numero_de_fatias, torcao=torcao)

    area_secao_1 = u3d.medir_area_do_poligono(secao_1)
    area_secao_2 = u3d.medir_area_do_poligono(secao_2)
    espessura = 4 / numero_de_fatias
    volume_por_fatias_1 = sum(u3d.medir_area_do_poligono(f) * espessura for f in fatias_1)
    volume_por_fatias_2 = sum(u3d.medir_area_do_poligono(f) * espessura for f in fatias_2)

    fig = u3d.criar_figura(titulo="Princípio de Cavalieri")
    if mostrar_fatias:
        fig = u3d.desenhar_pilha(fig, fatias_1, espessura)
        fig = u3d.desenhar_pilha(fig, fatias_2, espessura)
    else:
        fig = u3d.desenhar_poliedro(fig, *u3d.prisma(base_1, topo_1), opacidade=0.3)
        # Com 60 fatias bem finas, sem contorno, a pilha parece um sólido liso.
        fatias_finas = u3d.fatias_de_prisma(base_2, topo_2, 60, torcao=torcao)
        fig = u3d.desenhar_pilha(
            fig, fatias_finas, ALTURA_DOS_SOLIDOS / 60, cores=u3d.CORES["solido"],
            opacidade=0.3, mostrar_arestas=False,
        )
        # O plano de corte, horizontal, passando pelos dois sólidos.
        centro_do_corte = ((AFASTAMENTO + inclinacao) / 2, 0, altura_do_corte)
        fig = u3d.desenhar_plano(
            fig, centro_do_corte, (0, 0, 1), opacidade=0.1,
            tamanho=AFASTAMENTO + inclinacao + 4,
        )
        for secao in (secao_1, secao_2):
            fig = u3d.desenhar_poligono(
                fig, secao, cor=u3d.CORES["destaque"], opacidade=0.8
            )
    fig = u3d.adicionar_rotulo(fig, (0, 0, 4.6), "sólido 1")
    fig = u3d.adicionar_rotulo(fig, (AFASTAMENTO + inclinacao, 0, 4.6), "sólido 2")
    u3d.mostrar(fig)

    print(f"Altura do corte: {altura_do_corte:.2f}")
    print(f"  área da seção do sólido 1: {area_secao_1:.4f}")
    print(f"  área da seção do sólido 2: {area_secao_2:.4f}")
    print(f"Soma das fatias ({numero_de_fatias} fatias de espessura {espessura:.4f}):")
    print(f"  sólido 1: {volume_por_fatias_1:.4f}")
    print(f"  sólido 2: {volume_por_fatias_2:.4f}")
    prisma_reto = u3d.prisma(base_1, topo_1)
    print(f"Conferência: {u3d.medir_volume(*prisma_reto):.4f}")


interact(
    comparar_solidos,
    altura_do_corte=widgets.FloatSlider(
        min=0, max=3.75, step=0.25, value=2, description="altura do corte",
        continuous_update=False,
    ),
    inclinacao=widgets.FloatSlider(
        min=0, max=3, step=0.5, value=2, description="inclinação",
        continuous_update=False,
    ),
    torcao=widgets.FloatSlider(
        min=0, max=180, step=30, value=60, description="torção (°)",
        continuous_update=False,
    ),
    numero_de_fatias=widgets.IntSlider(
        min=4, max=60, step=4, value=12, description="fatias",
        continuous_update=False,
    ),
    mostrar_fatias=widgets.Checkbox(value=False, description="Mostrar fatias"),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='altura do corte', max=3.75,…

Nos prismas, as fatias se encaixam perfeitamente: a soma dá o volume **exato**, com qualquer número de fatias. Em sólidos com seções que mudam de tamanho (como pirâmides, cones e esferas), a soma das fatias é só uma **aproximação**, que fica melhor quanto mais finas forem as fatias.

> 📎 **Conexão.** Somar fatias cada vez mais finas é a ideia central do Cálculo, que você vai estudar no módulo 11. Por enquanto, o Princípio de Cavalieri nos dá respostas **exatas** sem precisar disso: basta comparar com um sólido cujo volume já conhecemos.

**Pare e pense**

1. Uma pilha de 100 moedas iguais, reta, e outra pilha com as mesmas 100 moedas, toda torta. Qual tem o maior volume?
2. Dois sólidos têm a mesma altura e bases de mesma área. Isso garante que têm o mesmo volume?

<details>
<summary>Ver respostas</summary>

1. Têm o mesmo volume: em cada altura, as seções são moedas de mesma área (Princípio de Cavalieri).
2. Não. É preciso que as seções tenham a mesma área em **todas** as alturas, não só na base.
</details>

## 4. Volume do prisma

**Teorema**
> O volume de um prisma é o produto da área da base pela altura:
> $$\boxed{V = S_b \cdot h}$$

**Demonstração (com Cavalieri).** Considere um prisma qualquer, de base com área $S_b$ e altura $h$, e um paralelepípedo retângulo com base de mesma área $S_b$ e a mesma altura $h$, ambos apoiados num mesmo plano $\alpha$.

- Todo plano paralelo a $\alpha$ que corta um dos sólidos corta o outro (eles têm a mesma altura).
- Num prisma, toda seção paralela à base é congruente à base (notebook 08). Então as seções dos dois sólidos têm área $S_b$.
- Pelo Princípio de Cavalieri, os volumes são iguais. O volume do paralelepípedo é $S_b \cdot h$ (seção 2). ∎

Isso vale para prismas **retos e oblíquos**, de **qualquer** base. Num prisma oblíquo de aresta lateral $\ell$ que forma ângulo $\theta$ com a base, $h = \ell \sin\theta$.

> 💡 **Dica.** Também é possível calcular o volume de um prisma oblíquo pela seção reta: $V = S_{\text{seção reta}} \cdot \ell$. É a mesma ideia da área lateral pela seção reta (notebook 08).

🕹️ Monte um prisma e compare o volume pela fórmula com a conferência.

In [ ]:
def calcular_volume_do_prisma(
    lados_da_base: int, lado: float, aresta_lateral: float, inclinacao: float
) -> None:
    """Desenha um prisma de base regular, com a altura, e calcula o seu volume."""
    base = u3d.poligono_regular(lados_da_base, lado=lado)
    # A aresta lateral "anda" ℓ · cos θ na horizontal e ℓ · sen θ na vertical
    # (razões trigonométricas), como no notebook 08.
    theta = math.radians(inclinacao)
    deslocamento = (aresta_lateral * math.cos(theta), 0, aresta_lateral * math.sin(theta))
    vertices, faces = u3d.prisma(base, deslocamento)

    apotema = lado / (2 * math.tan(math.pi / lados_da_base))
    area_base = lados_da_base * lado * apotema / 2
    altura = aresta_lateral * math.sin(math.radians(inclinacao))
    volume = area_base * altura

    fig = u3d.criar_figura(titulo="V = (área da base) × (altura)")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3)
    fig = u3d.desenhar_poligono(
        fig, base, rotulo="S_b", cor=u3d.CORES["plano"], opacidade=0.6
    )
    # A altura: do centro da base de baixo, subindo na vertical até o plano da base
    # de cima, onde forma um ângulo reto.
    pe, topo = (0, 0, 0), (0, 0, altura)
    fig = u3d.desenhar_segmento(
        fig, pe, topo, rotulo="h", cor=u3d.CORES["reta"], tracejado=True
    )
    fig = u3d.desenhar_angulo_reto(fig, topo, pe, (1, 0, altura), cor=u3d.CORES["reta"])
    if inclinacao < 90:
        # No prisma oblíquo, o topo da altura fica fora da base de cima: ligamos o
        # topo ao centro da base de cima, por dentro do plano dessa base.
        fig = u3d.desenhar_segmento(
            fig, topo, (deslocamento[0], 0, altura), cor=u3d.CORES["apoio"],
            espessura=2, tracejado=True,
        )
    u3d.mostrar(fig)

    print(f"Apótema da base: {apotema:.4f}")
    print(f"Área da base: S_b = (n · lado · apótema) / 2 = {area_base:.4f}")
    print(
        f"Altura: h = ℓ · sen θ = {aresta_lateral:g} · sen {inclinacao:g}° = {altura:.4f}"
    )
    print(f"Volume: V = S_b · h = {volume:.4f}")
    print(f"Conferência: {u3d.medir_volume(vertices, faces):.4f}")


interact(
    calcular_volume_do_prisma,
    lados_da_base=widgets.IntSlider(
        min=3, max=8, step=1, value=6, description="lados da base",
        continuous_update=False,
    ),
    lado=widgets.FloatSlider(
        min=1, max=3, step=0.5, value=2, description="lado", continuous_update=False
    ),
    aresta_lateral=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="aresta lateral",
        continuous_update=False,
    ),
    inclinacao=widgets.FloatSlider(
        min=30, max=90, step=10, value=90, description="inclinação (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=6, continuous_update=False, description='lados da base', max=8, min=3), …

**Pare e pense**

1. Um prisma hexagonal regular tem aresta da base 2 e altura 5. Qual é o volume?
2. Um prisma reto e um prisma oblíquo têm a mesma base e arestas laterais do mesmo comprimento. Qual tem o maior volume?

<details>
<summary>Ver respostas</summary>

1. $S_b = 6 \cdot \frac{2^2\sqrt{3}}{4} = 6\sqrt{3}$; $V = 6\sqrt{3} \cdot 5 = 30\sqrt{3} \approx 51{,}96$.
2. O reto. No oblíquo, a altura $\ell \sin\theta$ é menor que $\ell$, então o volume é menor.
</details>

## 5. Volumes no dia a dia

### O aquário e a pedra

Como medir o volume de uma pedra, que não tem forma de prisma? Coloque-a num aquário com água: o nível sobe. A água que "subiu" forma um prisma (um paralelepípedo) cuja base é o fundo do aquário e cuja altura é a subida do nível. **O volume da pedra é o volume desse paralelepípedo.**

> 🕵️ **Curiosidade histórica.** Conta a lenda que o rei Hierão, de Siracusa, pediu a **Arquimedes** (século III a.C.) que descobrisse se sua coroa era de ouro puro, sem danificá-la. Ao entrar numa banheira e ver a água subir, Arquimedes teria percebido como medir o volume da coroa e saído pelas ruas gritando "Eureka!" ("encontrei!"). A história provavelmente foi enfeitada ao longo dos séculos, mas a ideia de medir volumes pela água deslocada é real e muito usada até hoje.

### A água da chuva

A chuva é medida em **milímetros**: "choveu 25 mm" significa que, se a água não escorresse, formaria uma lâmina de 25 mm de altura sobre o chão. Sobre cada 1 m² de chão, isso é um paralelepípedo de $1 \text{ m} \times 1 \text{ m} \times 0{,}025 \text{ m} = 0{,}025 \text{ m}^3 = 25$ litros.
$$1 \text{ mm de chuva} = 1 \text{ litro por m}^2$$

**Captando a chuva do telhado.** Quanta água um telhado consegue captar para uma cisterna? Como a chuva cai na vertical, o que importa é a área da **projeção** do telhado no chão (a área da planta), e não a área do telhado inclinado (notebook 05). Um telhado mais inclinado tem mais área, mas capta a mesma quantidade de água!

🕹️ Use os dois widgets abaixo.

In [8]:
ALTURA_DO_AQUARIO = 40  # em centímetros


def colocar_pedra(
    comprimento_do_aquario: int, largura_do_aquario: int, nivel_inicial: int,
    aresta_da_pedra: int,
) -> None:
    """Coloca uma pedra cúbica no aquário e calcula quanto o nível da água sobe."""
    area_do_fundo = comprimento_do_aquario * largura_do_aquario
    volume_da_pedra = aresta_da_pedra ** 3
    subida = volume_da_pedra / area_do_fundo
    nivel_final = nivel_inicial + subida

    c, l = comprimento_do_aquario, largura_do_aquario
    fig = u3d.criar_figura(titulo="O aquário e a pedra")
    fig = u3d.desenhar_poliedro(
        fig, *u3d.paralelepipedo_retangulo(c, l, ALTURA_DO_AQUARIO), opacidade=0.1
    )
    # A água antes da pedra: um paralelepípedo com a altura do nível inicial.
    fig = u3d.desenhar_poliedro(
        fig, *u3d.paralelepipedo_retangulo(c, l, nivel_inicial), cor=u3d.CORES["plano"],
        opacidade=0.3, mostrar_arestas=False,
    )
    # A pedra (um cubo, para você poder conferir a conta) fica no meio do fundo.
    canto = ((c - aresta_da_pedra) / 2, (l - aresta_da_pedra) / 2, 0)
    fig = u3d.desenhar_blocos(
        fig, [canto], aresta=aresta_da_pedra, cor=u3d.CORES["plano_terciario"]
    )
    # A água que subiu: um paralelepípedo com a base do fundo e a altura da subida.
    camada = u3d.paralelepipedo_retangulo(c, l, subida, origem=(0, 0, nivel_inicial))
    fig = u3d.desenhar_poliedro(
        fig, *camada, cor=u3d.CORES["destaque"], opacidade=0.4, mostrar_arestas=False
    )
    u3d.mostrar(fig)

    print(f"Área do fundo: {c} × {l} = {area_do_fundo} cm²")
    print(
        f"Volume da pedra: {aresta_da_pedra}³ = {volume_da_pedra} cm³ "
        f"= {volume_da_pedra / 1000:g} litros"
    )
    print(f"Subida do nível: {volume_da_pedra} / {area_do_fundo} = {subida:.2f} cm")
    print(f"Nível final: {nivel_inicial} + {subida:.2f} = {nivel_final:.2f} cm")
    if aresta_da_pedra > nivel_final:
        print(
            "Atenção: a pedra não ficou totalmente coberta pela água. A conta só vale "
            "com a pedra submersa; aumente o nível inicial ou diminua a pedra."
        )


interact(
    colocar_pedra,
    comprimento_do_aquario=widgets.IntSlider(
        min=40, max=100, step=10, value=60, description="comprimento (cm)",
        continuous_update=False,
    ),
    largura_do_aquario=widgets.IntSlider(
        min=20, max=50, step=5, value=30, description="largura (cm)",
        continuous_update=False,
    ),
    nivel_inicial=widgets.IntSlider(
        min=10, max=30, step=2, value=20, description="nível inicial (cm)",
        continuous_update=False,
    ),
    aresta_da_pedra=widgets.IntSlider(
        min=4, max=16, step=2, value=12, description="aresta da pedra (cm)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=60, continuous_update=False, description='comprimento (cm)', min=40, ste…

In [9]:
ALTURA_DAS_PAREDES = 3  # em metros
EXAGERO_DA_CHUVA = 20  # a lâmina de chuva é desenhada 20 vezes mais alta, para ser vista


def captar_chuva(
    comprimento_da_planta: int, largura_da_planta: int, inclinacao_do_telhado: int,
    chuva: int,
) -> None:
    """Desenha a casa sob a chuva e calcula quanta água o telhado capta."""
    area_da_planta = comprimento_da_planta * largura_da_planta
    area_do_telhado = area_da_planta / math.cos(math.radians(inclinacao_do_telhado))
    litros_captados = area_da_planta * chuva          # 1 mm sobre 1 m² = 1 litro

    c, l, h = comprimento_da_planta, largura_da_planta, ALTURA_DAS_PAREDES
    meio = l / 2
    # Tangente (módulo 05): a cumeeira fica (largura / 2) · tan θ acima do beiral.
    cumeeira = h + meio * math.tan(math.radians(inclinacao_do_telhado))

    fig = u3d.criar_figura(titulo="Quanta chuva o telhado capta?", altura=550)
    # A casa (como no notebook 05): paredes, oitões e as duas águas do telhado.
    paredes = [
        [(0, 0, 0), (c, 0, 0), (c, 0, h), (0, 0, h)],
        [(c, 0, 0), (c, l, 0), (c, l, h), (c, 0, h)],
        [(c, l, 0), (0, l, 0), (0, l, h), (c, l, h)],
        [(0, l, 0), (0, 0, 0), (0, 0, h), (0, l, h)],
        [(0, 0, h), (0, l, h), (0, meio, cumeeira)],
        [(c, 0, h), (c, l, h), (c, meio, cumeeira)],
    ]
    for parede in paredes:
        fig = u3d.desenhar_poligono(fig, parede, opacidade=0.2)
    for lado in (0, l):
        agua = [(0, lado, h), (c, lado, h), (c, meio, cumeeira), (0, meio, cumeeira)]
        fig = u3d.desenhar_poligono(
            fig, agua, cor=u3d.CORES["plano_secundario"], opacidade=0.6
        )
    # A planta: a projeção do telhado no chão.
    planta = [(0, 0, 0), (c, 0, 0), (c, l, 0), (0, l, 0)]
    fig = u3d.desenhar_poligono(
        fig, planta, rotulo="planta", cor=u3d.CORES["destaque"], opacidade=0.3
    )

    # A lâmina de chuva, acima da casa, com a mesma base da planta. As linhas
    # tracejadas mostram que a chuva que cai sobre o telhado é a que cairia na planta.
    base_da_lamina = cumeeira + 1.5
    espessura_desenhada = chuva / 1000 * EXAGERO_DA_CHUVA
    lamina = u3d.paralelepipedo_retangulo(c, l, espessura_desenhada, origem=(0, 0, base_da_lamina))
    fig = u3d.desenhar_poliedro(fig, *lamina, cor=u3d.CORES["plano"], opacidade=0.4)
    for x, y, _ in planta:
        fig = u3d.desenhar_segmento(
            fig, (x, y, base_da_lamina), (x, y, 0), cor=u3d.CORES["apoio"],
            espessura=2, tracejado=True,
        )
    fig = u3d.adicionar_rotulo(
        fig, (c / 2, meio, base_da_lamina + espessura_desenhada + 0.8), f"chuva de {chuva} mm"
    )
    fig = u3d.definir_camera(fig, olho=(1.3, -1.6, 0.8))
    u3d.mostrar(fig)

    print(f"Área da planta: {c} × {l} = {area_da_planta} m²")
    print(
        f"Área do telhado: planta / cos {inclinacao_do_telhado}° = {area_do_telhado:.2f} m²"
    )
    print(f"Água captada: {area_da_planta} m² × {chuva} mm = {litros_captados} litros")
    print("A inclinação muda a área do telhado, mas não a quantidade de água captada.")


interact(
    captar_chuva,
    comprimento_da_planta=widgets.IntSlider(
        min=5, max=15, step=1, value=10, description="comprimento (m)",
        continuous_update=False,
    ),
    largura_da_planta=widgets.IntSlider(
        min=4, max=12, step=1, value=8, description="largura (m)",
        continuous_update=False,
    ),
    inclinacao_do_telhado=widgets.IntSlider(
        min=10, max=45, step=5, value=30, description="inclinação (°)",
        continuous_update=False,
    ),
    chuva=widgets.IntSlider(
        min=5, max=100, step=5, value=20, description="chuva (mm)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=10, continuous_update=False, description='comprimento (m)', max=15, min=…

**Pare e pense**

1. Choveu 10 mm numa praça de 500 m². Quantos litros de água caíram na praça?
2. Por que a quantidade de água captada não depende da inclinação do telhado?

<details>
<summary>Ver respostas</summary>

1. $500 \times 10 = 5000$ litros.
2. Porque a chuva cai na vertical: a água que atinge o telhado é a mesma que atingiria a sua projeção no chão.
</details>

## Resumo

**Volume:** quantas vezes o cubo unitário cabe no sólido. $1 \text{ m}^3 = 1000 \text{ L}$; $1 \text{ dm}^3 = 1 \text{ L}$; $1 \text{ cm}^3 = 1 \text{ mL}$. Em volume, cada passo de unidade multiplica por 1000.

**Paralelepípedo retângulo:** $V = abc = S_b \cdot h$. **Cubo:** $V = a^3$.

**Razão entre paralelepípedos retângulos:** $\dfrac{V_1}{V_2} = \dfrac{a_1 b_1 c_1}{a_2 b_2 c_2}$. Dobrar as três dimensões multiplica o volume por 8.

**Princípio de Cavalieri:** se todo plano paralelo a um plano dado corta os dois sólidos em seções de **mesma área**, os sólidos têm o **mesmo volume**.

**Prisma** (reto ou oblíquo): $V = S_b \cdot h$, com $h = \ell \sin\theta$ no oblíquo. Também $V = S_{\text{seção reta}} \cdot \ell$.

**Aplicações:** volume de objetos pela água deslocada; 1 mm de chuva = 1 L/m²; a água captada depende da área da **planta**.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0609b_volume_e_principio_de_cavalieri_gabarito.ipynb`](0609b_volume_e_principio_de_cavalieri_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: conversões de unidades. Escreva cada resposta como um número.
# (a) 1 m³ em litros;
# (b) 2,5 L em cm³;
# (c) 1 cm³ em mm³;
# (d) 3000 L em m³.
litros_ex1 = ...
cm3_ex1 = ...
mm3_ex1 = ...
m3_ex1 = ...

print(litros_ex1, cm3_ex1, mm3_ex1, m3_ex1)

In [ ]:
# Verificação
respostas_ex1 = [
    ("a", litros_ex1, 1000),
    ("b", cm3_ex1, 2500),
    ("c", mm3_ex1, 1000),
    ("d", m3_ex1, 3),
]
for item, resposta, certa in respostas_ex1:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), em volume, cada passo de unidade "
        "multiplica (ou divide) por 1000; 1 L = 1 dm³ (seção 1 - O que é volume)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: volumes. Calcule, em Python (use math.sqrt quando precisar):
# (a) o volume de um cubo de aresta 3;
# (b) o volume de um paralelepípedo retângulo 3 × 4 × 12;
# (c) o volume de um prisma hexagonal regular de aresta da base 2 e altura 5.
volume_cubo_ex2 = ...
volume_paralelepipedo_ex2 = ...
volume_prisma_ex2 = ...

print(volume_cubo_ex2, volume_paralelepipedo_ex2, volume_prisma_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", volume_cubo_ex2, 27, "seção 2 - Volume do paralelepípedo retângulo e do cubo"),
    ("b", volume_paralelepipedo_ex2, 144, "seção 2 - Volume do paralelepípedo retângulo e do cubo"),
    ("c", volume_prisma_ex2, 30 * math.sqrt(3), "seção 4 - Volume do prisma"),
]
for item, resposta, certa, secao in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), V = área da base × altura; a base "
        f"hexagonal regular é formada por 6 triângulos equiláteros ({secao})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Dobrar a aresta de um cubo dobra o seu volume.
# (b) Dois prismas com bases de mesma área e mesma altura têm o mesmo volume.
# (c) Um prisma oblíquo tem volume menor que um prisma reto de mesma base e mesma
#     aresta lateral.
# (d) Se dois sólidos têm o mesmo volume, então suas seções à mesma altura têm
#     sempre a mesma área.
respostas_ex3 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
}

print(respostas_ex3)

In [ ]:
# Verificação
gabarito_ex3 = {"a": False, "b": True, "c": True, "d": False}
dicas_ex3 = {
    "a": "dobrar a aresta dobra as três dimensões (seção 2 - Volume do paralelepípedo retângulo e do cubo)",
    "b": "V = S_b · h vale para qualquer prisma (seção 4 - Volume do prisma)",
    "c": "a altura do prisma oblíquo é ℓ · sen θ (seção 4 - Volume do prisma)",
    "d": "o erro comum de achar que o princípio vale a volta (seção 3 - O Princípio de Cavalieri)",
}
for item, certa in gabarito_ex3.items():
    assert respostas_ex3[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex3[item]}."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a caixa d'água. Uma caixa d'água tem a forma de um
# paralelepípedo retângulo de 1,5 m × 1,2 m × 1,0 m.
# (a) Quantos litros ela comporta?
# (b) Uma família de 4 pessoas gasta 150 litros por pessoa por dia. Quantos dias a
#     caixa cheia dura?
litros_ex4 = ...
dias_ex4 = ...

print(litros_ex4, dias_ex4)

In [ ]:
# Verificação
assert litros_ex4 is not ... and abs(litros_ex4 - 1800) <= 0.01, (
    "Tente de novo, revise: no item (a), calcule o volume em m³ e multiplique por "
    "1000 para ter litros (seção 1 - O que é volume)."
)
assert dias_ex4 is not ... and abs(dias_ex4 - 3) <= 0.01, (
    "Tente de novo, revise: no item (b), divida os litros da caixa pelo gasto da "
    "família em um dia, 4 × 150 litros (seção 1 - O que é volume)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o aquário. Um aquário tem fundo de 60 cm × 30 cm. Ao
# colocar uma pedra totalmente submersa, o nível da água sobe 2 cm. Qual é o volume
# da pedra:
# (a) em cm³;
# (b) em litros?
volume_cm3_ex5 = ...
volume_litros_ex5 = ...

print(volume_cm3_ex5, volume_litros_ex5)

In [ ]:
# Verificação
respostas_ex5 = [("a", volume_cm3_ex5, 3600), ("b", volume_litros_ex5, 3.6)]
for item, resposta, certa in respostas_ex5:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), a água que subiu forma um "
        "paralelepípedo de base 60 × 30 e altura 2; 1 L = 1000 cm³ "
        "(seção 5 - Volumes no dia a dia)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: a cisterna. Uma casa tem planta retangular de 8 m × 10 m
# e telhado com 30° de inclinação. Numa chuva de 25 mm, quantos litros o telhado
# capta?
largura_da_planta = 8
comprimento_da_planta = 10
inclinacao = 30
chuva = 25

litros_ex6 = ...

print(litros_ex6)

In [ ]:
# Verificação
assert litros_ex6 is not ... and abs(litros_ex6 - 2000) <= 0.01, (
    "Tente de novo, revise: use a área da planta (a projeção), não a área do "
    "telhado; 1 mm sobre 1 m² = 1 L (seção 5 - Volumes no dia a dia)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: cubo 10% maior. A aresta de um cubo aumenta 10%. Em quantos
# por cento aumenta o volume? (Responda só o número: para 50%, escreva 50.)
aumento_ex7 = ...

print(aumento_ex7)

In [ ]:
# Verificação
assert aumento_ex7 is not ... and abs(aumento_ex7 - 33.1) <= 0.01, (
    "Tente de novo, revise: o novo volume é (1,1a)³ = 1,331a³ "
    "(seção 2 - Volume do paralelepípedo retângulo e do cubo)."
)
print("Certo!")

In [ ]:
# Exercício Desafio 8: três faces, um volume. Três faces de um paralelepípedo
# retângulo, com um vértice em comum, têm áreas 12 cm², 15 cm² e 20 cm². Qual é o
# volume?
# Dica: se as dimensões são a, b, c, as áreas são ab, ac e bc. O que acontece quando
# você multiplica as três?
volume_ex8 = ...

print(volume_ex8)

In [ ]:
# Verificação
assert volume_ex8 is not ... and abs(volume_ex8 - 60) <= 0.01, (
    "Tente de novo, revise: (ab) · (ac) · (bc) = (abc)², então V = √(12 · 15 · 20) "
    "(seção 2 - Volume do paralelepípedo retângulo e do cubo)."
)
print("Certo!")
print("As dimensões são 3, 4 e 5. Confira: 3 · 4 = 12, 3 · 5 = 15, 4 · 5 = 20.")

## Conexões

**Isso depende de:**
- [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (elementos, altura e seções do prisma)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (área da projeção)
- [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (áreas de polígonos)

**Isso será usado em:**
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (volume da pirâmide, por Cavalieri e pela decomposição do prisma)
- [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb) (volume do cilindro comparado ao do prisma, por Cavalieri)
- 📌 Cone e 📌 Esfera (volumes por Cavalieri)
- 📌 Sólidos semelhantes e troncos (razão entre volumes de sólidos semelhantes)
- 📌 Limite, Integral e Derivada (volumes por soma de fatias)